In [ ]:
import os
import pickle
from collections import Counter

import datamapplot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bertopic import BERTopic
from forestplot import forestplot
from hdbscan import HDBSCAN
from matplotlib.ticker import FuncFormatter
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

plt.rcParams["font.monospace"] = ["Noto Mono", "Tlwg Mono", "Courier New"]

from helpers import save_mpl_fig

In [ ]:
df = (
    pd.read_parquet("gc_human_tweets.parquet")
    # Remove "metoo" before topic
    # (will become spam when bertopic concatenates tweets as one doc)
    .assign(
        preprocessed_content=lambda x: x["preprocessed_content"]
        .str.replace("#metoo", "", regex=False, case=False)
        .str.replace("metoo", "", regex=False, case=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    # Filter out very short tweets
    .assign(word_count=lambda x: x["preprocessed_content"].str.split().str.len())
    .query("word_count >= 3")
    .drop(columns=["word_count"])
    .assign(
        month=lambda df_: pd.to_datetime(df_["date"])
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    #     .sample(n=100_000, random_state=0)
)
df

## Get topics

In [ ]:
# 'all-mpnet-base-v2' is a high-quality general-purpose model that balances
# performance and speed, producing 768-dimensional dense embeddings
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# UMAP for dimensionality reduction
umap_model = UMAP(
    # number of neighboring sample points (2 to 100)
    # higher = more global structure, lower = more local (15-50 typical)
    n_neighbors=30,
    # dimension of the space to embed into (2 to 100)
    # higher = higher dimension/harder to cluster later
    n_components=5,
    # Min distance between points; higher = more spread out
    # 0.0 = tight packing for distinct clusters
    min_dist=0.0,
    # metric to compute distances in high dimensional space
    metric="cosine",
    random_state=42,
)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    # Min samples to form cluster
    # higher = fewer/larger topics
    min_cluster_size=300,
    # Distance metric; standard for UMAP-reduced space
    metric="euclidean",
    # Standard for HDBSCAN
    cluster_selection_method="eom",
    prediction_data=False,
    # number of samples in a neighborhood for a point to be considered as a core point
    # https://maartengr.github.io/BERTopic/faq.html#how-do-i-reduce-topic-outliers
    # lower = fewer outliers, but more noise in non-outlier topics
    min_samples=100,
)

# CountVectorizer for topic representation
vectorizer_model = CountVectorizer(
    ngram_range=(1, 1),
    stop_words="english",  # Remove common English stopwords
    max_features=None,
)

# Init BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    language="english",
    calculate_probabilities=False,
    verbose=True,
)
print("Initialized BERTopic")

In [ ]:
embeddings_path = "./output/embeddings_768d.npy"

if os.path.exists(embeddings_path):
    embeddings = np.load(embeddings_path)
    print(f"Loaded embeddings: {embeddings.shape}")
else:
    print("Computing embeddings (this will take ~2hr)...")
    embeddings = embedding_model.encode(
        df["preprocessed_content"].tolist(), show_progress_bar=True
    )
    np.save(embeddings_path, embeddings)
    print(f"Computed and saved embeddings: {embeddings.shape}")

In [ ]:
topics_path = "./output/bertopics.parquet"
model_path = "./output/topic_model.pkl"

# Check if both topics AND model exist
if os.path.exists(topics_path) and os.path.exists(model_path):
    print("Loading existing topics and model...")
    df = pd.read_parquet(topics_path)
    topic_model = pickle.load(open(model_path, "rb"))
    print(
        f"✓ Loaded {len(df):,} tweets with {len(topic_model.get_topic_info())-1} topics"
    )
else:
    print("Computing topics (this will take ~15 min)...")
    topics, _ = topic_model.fit_transform(
        df["preprocessed_content"], embeddings=embeddings
    )
    df["topic"] = topics

    # Save both
    df.to_parquet(topics_path, index=False)
    pickle.dump(topic_model, open(model_path, "wb"))
    print("✓ Saved topics and model")

## Eval

In [ ]:
print("\n Listing topics...")


def analyze_topics(topic_model, df):
    """Generate comprehensive topic analysis"""

    # Get topic info
    topic_info = topic_model.get_topic_info()

    print("\n" + "=" * 80)
    print("TOPIC SUMMARY")
    print("=" * 80)
    print(f"\nTotal topics discovered: {len(topic_info) - 1}")  # -1 for outlier topic
    print(f"Total tweets: {len(df):,}")
    print(
        f"Outlier tweets (topic -1): {topic_info[topic_info['Topic'] == -1]['Count'].values[0]:,}"
    )

    # Display top topics
    _n = 30
    print("\n" + "-" * 80)
    print(f"TOP {_n} TOPICS BY SIZE")
    print("-" * 80)

    for idx, row in topic_info.head(_n).iterrows():
        topic_id = row["Topic"]
        count = row["Count"]
        pct = (count / len(df)) * 100

        # Get representative words
        words = topic_model.get_topic(topic_id)[:10]
        word_str = ", ".join([word for word, score in words])

        print(f"\nTopic {topic_id}: {count:,} tweets ({pct:.1f}%)")
        print(f"  Top words: {word_str}")

    return topic_info


topic_info = analyze_topics(topic_model, df)

### Labels

In [ ]:
# Manual labels for top topics
TOPIC_LABELS = {
    -1: "Core MeToo discourse",
    0: "Kavanaugh hearings (SCOTUS)",
    1: "Political co-opting",
    2: "Hollywood/Film",
    3: "Church abuse",
    4: "Music industry",
    5: "Tarana Burke",
    6: "Men, masculinity & feminism",
    7: "Movement discourse",
    8: "Workplace harassment",
    9: "Golden Globes/Oprah",
    10: "Academia/campus misconduct",
    11: "Social media meta",
    12: "Rape & assault",
    13: "Daily news",
    14: "Ford testimony (SCOTUS)",
    15: "Harvey Weinstein",
    16: "Clinton/Lewinsky",
    17: "Race & BLM",
    18: "Survivors, healing/support",
    19: "Bill Cosby",
    20: "Democratic scandals",
    21: "Women's March",
    22: "Aziz Ansari",
    23: "Poetry/writing",
}

n_topics = df["topic"].value_counts()
n_tweets = len(df)

# Manual labels with size annotations
TOPIC_LABELS_WITH_SIZE = {}
for topic_id, label in TOPIC_LABELS.items():
    count = n_topics.get(topic_id, 0)
    pct = (count / n_tweets) * 100
    TOPIC_LABELS_WITH_SIZE[topic_id] = f"{label}\n({pct:.1f}%)"

# Create labels list
labels = []
for t in df["topic"]:
    if t in TOPIC_LABELS_WITH_SIZE:
        labels.append(TOPIC_LABELS_WITH_SIZE[t])
    else:
        labels.append(TOPIC_LABELS_WITH_SIZE[-1])

## Viz

### Prep

In [ ]:
embeddings_2d_path = "./output/embeddings_2d.npy"

if os.path.exists(embeddings_2d_path):
    reduced_embeddings_2d = np.load(embeddings_2d_path)
    print(f"Loaded 2D embeddings: {reduced_embeddings_2d.shape}")
else:
    print("Computing 2D UMAP... (will take ~20mins)")
    umap_2d = UMAP(
        n_neighbors=20, n_components=2, min_dist=0.0, metric="cosine", random_state=42
    )
    reduced_embeddings_2d = umap_2d.fit_transform(embeddings)
    np.save(embeddings_2d_path, reduced_embeddings_2d)
    print(f"Computed and saved 2D embeddings: {reduced_embeddings_2d.shape}")

In [ ]:
mapplot_opts = dict(
    labels=labels,
    figsize=(20, 15),
    label_font_size=14,
    font_weight=900,
    label_wrap_width=10,
    use_medoids=True,
    dpi=200,
    #     label_over_points=True,
)

### Dark

In [ ]:
_, ax = datamapplot.create_plot(
    reduced_embeddings_2d,
    **mapplot_opts,
    darkmode=True,
    color_label_text=".8",
)

x_min, x_max = np.percentile(reduced_embeddings_2d[:, 0], [2, 98])
y_min, y_max = np.percentile(reduced_embeddings_2d[:, 1], [2, 98])
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
# ax.axis("off")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)
plt.tight_layout()
save_mpl_fig("./output/figures/topic_2Dmap_darkmode")

### Light

In [ ]:
_, ax = datamapplot.create_plot(
    reduced_embeddings_2d,
    **mapplot_opts,
    color_label_text=".5",
)

x_min, x_max = np.percentile(reduced_embeddings_2d[:, 0], [2, 95])
y_min, y_max = np.percentile(reduced_embeddings_2d[:, 1], [5, 97])
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
# ax.axis("off")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)
plt.tight_layout()
save_mpl_fig("./output/figures/topic_2Dmap")

## HTML

Todo

* remove unlabelled topic labels in the map
* Add avg citations?
* Add a Tweet: header just before the tweet?
* also is it me or are the points/markers smaller now...
* bold keywords header

### Prep

In [ ]:
# =======================================================================
# Compute avg RTs and favs per topic
topic_stats = (
    df.groupby("topic").agg({"retweets": "mean", "favorites": "mean"}).reset_index()
)
topic_stats.columns = ["topic", "avg_rt", "avg_fav"]

topic_stats_dict = {
    row["topic"]: (row["avg_rt"], row["avg_fav"]) for _, row in topic_stats.iterrows()
}

n_topics = df["topic"].value_counts()
n_tweets = len(df)

# =======================================================================
# Populate hover data
hover_texts = []
extra_data_rows = []

for idx, row in df.iterrows():
    topic_id = row["topic"]

    if topic_id in TOPIC_LABELS:
        topic_label = TOPIC_LABELS[topic_id]
    else:
        topic_label = f"Topic {topic_id}"

    # Get keywords
    if topic_id != -1:
        words = topic_model.get_topic(topic_id)[:5]
        keywords = ", ".join([w for w, _ in words])
    else:
        outlier_words = topic_model.get_topic(-1)[:5]
        keywords = ", ".join([w for w, _ in outlier_words])

    count = n_topics.get(topic_id, 0)
    pct = (count / n_tweets) * 100
    content = row["content"]

    # Get topic stats
    avg_rt, avg_fav = topic_stats_dict.get(topic_id, (0, 0))

    hover_texts.append(content)

    extra_data_rows.append(
        {
            "topic_label": topic_label,
            "topic_id": str(topic_id),
            "count": f"{count:,}",
            "pct": f"{pct:.1f}",
            "keywords": keywords,
            "avg_rt": f"{avg_rt:.1f}",
            "avg_fav": f"{avg_fav:.1f}",
        }
    )

df_topic_metadata = pd.DataFrame(extra_data_rows)


# =======================================================================
# Get labeled topics
def get_topic_label_clean(topic_id):
    if topic_id in TOPIC_LABELS:
        return TOPIC_LABELS[topic_id]
    else:
        return ""


topic_label_list = [get_topic_label_clean(t) for t in df["topic"]]

# Get unique labeled topic IDs (sorted for consistency)
labeled_topic_ids = sorted([tid for tid in TOPIC_LABELS.keys() if tid != -1])

# =======================================================================
# make colors (markers + leged)
# Get all colors
cmap_tab20b = plt.cm.tab20b
cmap_tab20c = plt.cm.tab20c

all_colors = []
for i in range(20):
    all_colors.append(cmap_tab20b(i / 20))
    all_colors.append(cmap_tab20c(i / 20))

# Shuffle for maximum distinction
np.random.seed(42)  # For reproducibility
shuffled_indices = np.random.permutation(len(all_colors))
shuffled_colors = [all_colors[i] for i in shuffled_indices]
n_colors = len(labeled_topic_ids)
colors = shuffled_colors[:n_colors]

# n_colors = len(labeled_topic_ids)
# colors = [cmap(i / n_colors) for i in range(n_colors)]
color_hex = [
    "#%02x%02x%02x" % (int(r * 255), int(g * 255), int(b * 255))
    for r, g, b, _ in colors
]

# Create color dict: topic_id -> color
topic_color_dict = dict(zip(labeled_topic_ids, color_hex))

# Create marker color array (one color per tweet)
marker_color_array = []
for topic_id in df["topic"]:
    if topic_id in topic_color_dict:
        marker_color_array.append(topic_color_dict[topic_id])
    else:
        marker_color_array.append("#d0d0d0")  # Gray for unlabeled/outliers

### html css

In [ ]:
# =======================================================================
# Hover text styling
badge_css = """
    border-radius:6px;
    width:fit-content;
    max-width:85%;
    margin:2px;
    padding: 2px 10px 2px 10px;
    font-size: 10pt;
"""

hover_text_template = f"""
<div>
    <div style="font-size:13pt;padding:4px;font-weight:bold;">{{topic_label}}</div>
    <div style="padding:6px 2px 2px 2px;font-size:10pt;color:#555;"><b>Tweet:</b><br>{{hover_text}}</div>
    <div style="background-color:#3498db;color:#fff;{badge_css}">Topic {{topic_id}}</div>
    <div style="background-color:#ecf0f1;color:#2c3e50;{badge_css}"><b>Keywords:</b> {{keywords}}</div>
    <div style="background-color:#2ecc71;color:#fff;{badge_css}">{{count}} tweets ({{pct}}%)</div>
    <div style="background-color:#e67e22;color:#fff;{badge_css}">Avg: {{avg_rt}} RTs, {{avg_fav}} ♥</div>
</div>
"""

# =======================================================================
# Legend CSS
custom_css = """
.legend-row {
    display: flex;
    align-items: center;
    margin-bottom: 5px;
    cursor: pointer;
}
.legend-row:hover {
    background-color: #f0f0f0;
}
.legend-box {
    height: 12px;
    width: 12px;
    border-radius: 3px;
    margin-right: 8px;
    flex-shrink: 0;
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-size: 8pt;
    font-weight: bold;
}
#legend {
    position: absolute;
    top: 10px;
    right: 10px;
    background-color: rgba(255, 255, 255, 0.95);
    padding: 15px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
    max-height: 80vh;
    overflow-y: auto;
    z-index: 2;
    font-size: 10pt;
    min-width: 200px;
}
#legend-title {
    font-weight: bold;
    margin-bottom: 10px;
    font-size: 12pt;
    border-bottom: 1px solid #ddd;
    padding-bottom: 5px;
}
"""

custom_js = """
const legend = document.getElementById("legend");
const selectedTopics = new Set();

legend.addEventListener('click', function (event) {
    let target = event.target;
    
    if (target.classList.contains('legend-box') || target.tagName === 'SPAN') {
        const row = target.closest('.legend-row');
        const box = row.querySelector('.legend-box');
        const topicLabel = row.querySelector('span').textContent;
        
        if (selectedTopics.has(topicLabel)) {
            selectedTopics.delete(topicLabel);
            box.innerHTML = "";
        } else {
            selectedTopics.add(topicLabel);
            box.innerHTML = "✓";
        }
        
        const selectedIndices = [];
        if (datamap.metaData.topic_label) {
            datamap.metaData.topic_label.forEach((label, i) => {
                if (selectedTopics.has(label)) {
                    selectedIndices.push(i);
                }
            });
        }
        
        datamap.addSelection(selectedIndices, "legend");
    }
});
"""

# =======================================================================
# Legend HTML using same colors...
# otherwise colors tagged to legend and on the map will be different..
custom_html = """
<div id="legend">
    <div id="legend-title">Top topics</div>
"""

for tid in labeled_topic_ids:
    label = TOPIC_LABELS[tid]
    color = topic_color_dict[tid]
    custom_html += f'    <div class="legend-row"><div class="legend-box" style="background-color:{color};"></div><span>{label}</span></div>\n'

custom_html += """
</div>
"""

### Plot

In [ ]:
# =======================================================================
df["month"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

plot = datamapplot.create_interactive_plot(
    reduced_embeddings_2d,
    topic_label_list,
    hover_text=hover_texts,
    extra_point_data=df_topic_metadata,
    hover_text_html_template=hover_text_template,
    title="Topic Map of #metoo Tweets",
    #     sub_title=f"({len(df):,} tweets, {len(topic_info)} topics)",
    sub_title=f"""2018 #MeToo tweets<br>(n = {len(df):,}, {len(topic_info)} topics)<br>
    <span style='font-size:9pt; color:#666;'>
    Embeddings: sentence-transformers/all-mpnet-base-v2 (768D) |
    UMAP: n_neighbors=30, n_components=5 | <br> 
    HDBSCAN: min_cluster_size=300, min_samples=50 | 
    Viz: UMAP 2D projection
    </span>""",
    enable_search=True,
    initial_zoom_fraction=0.85,
    marker_color_array=marker_color_array,
    custom_css=custom_css,
    custom_html=custom_html,
    custom_js=custom_js,
    point_radius_min_pixels=0.5,
    point_radius_max_pixels=500,
    point_line_width=0,
    histogram_data=df["month"],
    histogram_group_datetime_by="month",
    histogram_range=(pd.to_datetime("2018-01-01"), pd.to_datetime("2018-12-31")),
    #     darkmode=True,
)

plot.save("./output/topic_2Dmap.html")
plot

## Tabulate

In [ ]:
topic_metadata = []
N_TOPICS = 25
TOPICS_LIST = range(-1, N_TOPICS - 1)
for topic_id in TOPICS_LIST:
    # Get tweets in this topic
    topic_tweets = df[df["topic"] == topic_id]

    if len(topic_tweets) == 0:
        continue

    # Basic info
    count = len(topic_tweets)
    pct = (count / len(df)) * 100

    # Get top 5 words
    words = topic_model.get_topic(topic_id)[:5]
    top_words = ", ".join([word for word, _ in words])

    # Calculate engagement
    mean_retweets = topic_tweets["retweets"].mean()
    mean_favs = topic_tweets["favorites"].mean()

    # Calculate mean hashtags per tweet in this topic
    hashtag_counts = topic_tweets["hashtags"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    mean_hashtags = hashtag_counts.mean()

    # Get top 3 hashtags in this topic
    all_hashtags = []
    for hashtag_list in topic_tweets["hashtags"]:
        if isinstance(hashtag_list, list):
            all_hashtags.extend(hashtag_list)

    if all_hashtags:
        top_3_hashtags = [tag for tag, _ in Counter(all_hashtags).most_common(4)]
        # Remove the # symbol for cleaner display
        top_3_hashtags_clean = [tag.replace("#", "") for tag in top_3_hashtags[1:4]]
        top_hashtags_str = ", ".join(top_3_hashtags_clean)
    else:
        top_hashtags_str = "—"

    topic_metadata.append(
        {
            "topic_num": topic_id,
            "count": count,
            "pct": pct,
            "top_words": top_words,
            "top_hashtags": top_hashtags_str,
            "mean_hashtags": mean_hashtags,  # Added this
            "mean_retweets": mean_retweets,
            "mean_favs": mean_favs,
        }
    )

df_plot = pd.DataFrame(topic_metadata)
df_plot

In [ ]:
def truncate_words_intelligently(word_score_list, max_len=70):
    """
    Build a string from word list up to max_len characters,
    only including complete words (no truncation mid-word).

    Parameters:
    -----------
    word_score_list : list of tuples
        List of (word, score) tuples from topic_model.get_topic()
    max_len : int
        Maximum character length for the final string

    Returns:
    --------
    str : Comma-separated words that fit within max_len
    """
    words = []
    current_len = 0

    for word, score in word_score_list:
        # Calculate what the length would be if we add this word
        # Account for ", " separator (2 chars) except for first word
        separator_len = 2 if words else 0
        new_len = current_len + separator_len + len(word)

        # If adding this word would exceed max_len, stop
        if new_len > max_len:
            break

        words.append(word)
        current_len = new_len

    return ", ".join(words)


topic_metadata = []
for topic_id in TOPICS_LIST:
    # Get tweets in this topic
    topic_tweets = df[df["topic"] == topic_id]

    if len(topic_tweets) == 0:
        continue

    # Basic info
    count = len(topic_tweets)
    pct = (count / len(df)) * 100

    # Get ALL words (not just top 5), then intelligently truncate
    all_words = topic_model.get_topic(topic_id)  # Gets all words
    top_words = truncate_words_intelligently(all_words, max_len=50)

    # Calculate engagement
    mean_retweets = topic_tweets["retweets"].mean()
    mean_favs = topic_tweets["favorites"].mean()

    # Calculate mean hashtags per tweet in this topic
    hashtag_counts = topic_tweets["hashtags"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    mean_hashtags = hashtag_counts.mean()

    #     # Calculate mean tweet len
    #     tweet_len = topic_tweets['retweets'].mean()

    topic_metadata.append(
        {
            "topic_num": topic_id,
            "count": count,
            "pct": pct,
            "top_words": top_words,
            "mean_hashtags": mean_hashtags,
            "mean_retweets": mean_retweets,
            "mean_favs": mean_favs,
        }
    )

df_plot = pd.DataFrame(topic_metadata)
df_plot

In [ ]:
df_plot["label"] = TOPIC_LABELS.values()

df_plot["count_str"] = df_plot["count"].apply(lambda x: f"{x:,}".rjust(7))
df_plot["pct_str"] = df_plot["pct"].apply(lambda x: f"{x:.1f}%".rjust(8))
df_plot["rt_str"] = df_plot["mean_retweets"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["fav_str"] = df_plot["mean_favs"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["mean_hashtags_str"] = df_plot["mean_hashtags"].apply(
    lambda x: f"{x:.1f}".rjust(8)
)

# Pad the top_words column to ensure consistent spacing
df_plot["top_words_padded"] = df_plot["top_words"].apply(
    lambda x: x.rjust(50) if len(x) < 50 else x
)

# For y-axis label: just the number
df_plot["topic_label"] = df_plot["topic_num"].astype(str)

# For forestplot
df_plot["ll"] = df_plot["pct"]
df_plot["hl"] = df_plot["pct"]

df_plot.loc[0, "pct"] = np.nan
df_plot.loc[0, "ll"] = np.nan
df_plot.loc[0, "hl"] = np.nan
# df_plot.loc[0, "count_str"] = df_plot.loc[0, "count_str"].rjust(0)
df_plot

In [ ]:
# plot
ax = forestplot(
    df_plot,
    estimate="pct",
    ll="ll",
    hl="hl",
    varlabel="label",
    annote=["top_words", "rt_str", "fav_str"],
    annoteheaders=["Topic (top terms)", "  RT", "Fave"],
    rightannote=["count_str", "pct_str"],
    right_annoteheaders=[" Tweets", "Tweets(%)"],
    ci_report=False,
    figsize=(2.5, N_TOPICS * 0.4),
    color_alt_rows=True,
    table=True,
    xticks=range(0, 5),
    **{
        "marker": "o",
        "variable_header": "Label",
        "markersize": 40,
        "xlinecolor": ".8",
        "xtick_size": 12,
    },
)

# Clean up axes
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{int(x)}%"))

ax.set_xlabel("Percentage of sample", fontsize=12)
for vline in [1, 2, 3, 4]:
    ax.vlines(vline, ymin=-0.5, ymax=N_TOPICS - 0.5, colors=".85", linewidth=0.5)

# for tick in ax.get_yticklabels():
#     tick.set_fontsize(11)
save_mpl_fig("./output/figures/top_topics_tabulation")